# 01 · FoodNExTDB download, schema and image audit

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Requires actual supervisor execution and research-use terms decisions. The adapter checks the published schema and stops rather than guessing malformed records.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Check the public-data track and permission records

In [ ]:
from oncoplate.governance import require_gate
assert cfg['study']['dataset']=='foodnextdb', "This notebook is for the separate FoodNExTDB foundation."
require_gate(cfg,'supervisor_execution');require_gate(cfg,'foodnextdb_research_terms')
initialize(cfg)

## 2. Download once, stage locally, audit
The ZIP remains in Drive; extracted small files are read from Colab local storage. The actual SHA-256 is saved; no publisher checksum is invented.

In [ ]:
from pathlib import Path
import pandas as pd
from oncoplate.config import paths

p = paths(cfg)
miss = pd.read_csv(p["prepared"] / "missing_images.csv", dtype=str)
print(miss.to_string(), "\n")

EXT = (".jpg", ".jpeg", ".png")
root = p["local"] / "extracted"
by_name = {}
by_stem = {}
for f in root.rglob("*"):
    if f.suffix.lower() in EXT:
        by_name[f.name] = f
        by_stem.setdefault(f.stem.lower(), []).append(f.name)

print("images on disk:", len(by_name), "\n")
for iid in miss["image_id"]:
    stem = Path(iid).stem.lower()
    near = by_stem.get(stem, [])
    same_folder = sorted(x for x in by_name if x.startswith(iid.split("_")[0] + "_" + iid.split("_")[1]))[:3]
    print(f"{iid:32s} exact:{iid in by_name}  same-stem:{near}  siblings:{same_folder}")

In [ ]:
import zipfile
from oncoplate.config import paths

arc = paths(cfg)["raw"] / "FoodNExtDB.zip"
want = {"A4F_69129_0019.jpg", "A4F_74720_0031.jpg"}
with zipfile.ZipFile(arc) as z:
    names = z.namelist()
    present = {n.split("/")[-1] for n in names}
    for w in sorted(want):
        hits = [n for n in names if n.endswith(w)]
        print(f"{w}: in archive = {w in present}  entries={hits}")
    for pid in ("A4F_69129", "A4F_74720"):
        imgs = sorted(n.split("/")[-1] for n in names
                      if f"/{pid}/" in n and n.lower().endswith(".jpg"))
        print(f"\n{pid}: {len(imgs)} images, last 5 = {imgs[-5:]}")

In [ ]:
from pathlib import Path

p = Path("/content/drive/MyDrive/OncoPlate_Research/oncoplate-research/src/oncoplate/datasets.py")
src = p.read_text()

old = """    if missing:
        write_table(output/"missing_images.csv",pd.DataFrame(missing))
        raise ValueError(f"{len(missing)} annotation records lack images; see missing_images.csv")"""

new = """    excluded_missing=[]
    if missing:
        # Upstream archive gap: the published CSVs reference images absent from the
        # distributed ZIP. Excluded here and reported; never silently dropped.
        write_table(output/"missing_images.csv",pd.DataFrame(missing))
        excluded_missing=sorted({m["image_id"] for m in missing})
        if len(excluded_missing)>MAX_MISSING_IMAGES:
            raise ValueError(f"{len(excluded_missing)} distinct images missing; exceeds tolerance {MAX_MISSING_IMAGES}")
        rows=[r for r in rows if r["record_id"] not in set(excluded_missing)]"""

assert src.count(old) == 1, f"expected 1 match, found {src.count(old)}"
src = src.replace(old, new)

old2 = """            "protocol_confirmation_required":True,"""
new2 = """            "protocol_confirmation_required":True,
            "excluded_missing_images":excluded_missing,
            "excluded_missing_image_count":len(excluded_missing),
            "excluded_annotation_rows":len(missing),"""
assert src.count(old2) == 1, f"expected 1 match, found {src.count(old2)}"
src = src.replace(old2, new2)

if "MAX_MISSING_IMAGES" not in src.split("def audit_foodnextdb")[0]:
    src = src.replace("def audit_foodnextdb", "MAX_MISSING_IMAGES=20\n\n\ndef audit_foodnextdb", 1)

p.write_text(src)
print("patched:", len(p.read_bytes()), "bytes")

In [ ]:
# after applying the patch: Runtime -> Restart session, then run notebook 01's setup cell, then this
from oncoplate.pipeline import audit_public
import json

report = audit_public(cfg, download_missing=False)
print(json.dumps(report, indent=2))

## 3. Inspect actual annotation fields and vocabulary

In [ ]:
records=read_table(p['prepared']/"records.csv")
ann=read_table(p['prepared']/"annotations.csv")
display(records.head());display(ann.head())
print(json.dumps(read_json(p['prepared']/"vocabulary_audit.json"),indent=2))

## 4. Check a real image and its panel records
No bounding boxes or cross-reviewer item identity are inferred.

In [ ]:
from PIL import Image
from IPython.display import display
rid=records.record_id.iloc[0]
display(Image.open(records.loc[records.record_id.eq(rid),'image_path'].iloc[0]).resize((480,480)))
display(ann[ann.record_id.eq(rid)])

## 5. Generate near-duplicate proposals for human review
An edge is joined only when `approved` is explicitly true. Unreviewed similarity is not proof of duplication.

In [ ]:
from oncoplate.splits import propose_near_duplicates
edges=propose_near_duplicates(records,cfg['data']['near_duplicate_hamming_distance'])
review_path=p['prepared']/"near_duplicate_review.csv"
if not review_path.exists():write_table(review_path,edges)
print("Review:",review_path,"| proposed pairs:",len(edges))
print("Use the decision file described in notebook 03 to document completed review; do not split near-duplicate families apart.")

In [ ]:
MESSAGE = "nb01: FoodNExTDB audit passed; 9262 images, 97 participants, 2 upstream-missing images excluded"

import sys, subprocess
r = subprocess.run([sys.executable, "tools/commit_cell.py", MESSAGE],
                   cwd="/content/drive/MyDrive/OncoPlate_Research/oncoplate-research",
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr, file=sys.stderr); print("exit:", r.returncode)

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
